In [ ]:

import os
import logging

from langsmith import Client
from langchain_groq import ChatGroq

logger = logging.getLogger(__name__)




In [ ]:
class RAGEvaluator:

    def __init__(
        self,
        dataset_name="Document-RAG-Evaluation",
        model="openai/gpt-oss-120b"
    ):

        self.dataset_name = dataset_name

        self.client = Client()

        self.judge = ChatGroq(
            model=model,
            temperature=0,
            max_tokens=500
        )

        if not os.getenv("LANGSMITH_API_KEY"):
            raise ValueError("LANGSMITH_API_KEY is missing")

    def create_dataset(self, examples):

        if self.client.has_dataset(
            dataset_name=self.dataset_name
        ):
            logger.info("Dataset already exists")
            return

        dataset = self.client.create_dataset(
            dataset_name=self.dataset_name,
            description="Dataset for RAG evaluation"
        )

        self.client.create_examples(
            dataset_id=dataset.id,
            examples=examples
        )

        logger.info("Evaluation dataset created")

    def _judge_score(self, prompt):

        response = self.judge.invoke(prompt)

        result = response.content.strip().upper()

        return result.startswith("YES")


    def correctness(self, inputs, outputs, reference_outputs):

        prompt = f"""
        Evaluate the factual correctness of the generated answer.

        Question:
        {inputs['question']}

        Expected answer:
        {reference_outputs['answer']}

        Generated answer:
        {outputs['answer']}

        Is the generated answer factually correct
        and consistent with the expected answer?

        Return only YES or NO.
        """

        score = self._judge_score(prompt)

        return {
            "key": "correctness",
            "score": score
        }


    def answer_relevance(self, inputs, outputs):

        prompt = f"""
        Evaluate whether the generated answer
        directly addresses the user's question.

        Question:
        {inputs['question']}

        Generated answer:
        {outputs['answer']}

        Return YES if relevant, otherwise NO.
        """

        score = self._judge_score(prompt)

        return {
            "key": "answer_relevance",
            "score": score
        }

    def faithfulness(self, inputs, outputs):

        prompt = f"""
        Evaluate whether the generated answer
        is supported by the retrieved context.

        Retrieved context:
        {outputs['context']}

        Generated answer:
        {outputs['answer']}

        Are all factual claims supported by
        the retrieved context?

        Return only YES or NO.
        """

        score = self._judge_score(prompt)

        return {
            "key": "faithfulness",
            "score": score
        }


    def retrieval_relevance(self, inputs, outputs):

        prompt = f"""
        Evaluate whether the retrieved documents
        contain useful information for answering
        the user's question.

        Question:
        {inputs['question']}

        Retrieved context:
        {outputs['context']}

        Return YES if relevant information exists,
        otherwise NO.
        """

        score = self._judge_score(prompt)

        return {
            "key": "retrieval_relevance",
            "score": score
        }

    def evaluate(self, rag_pipeline):

        def target(inputs: dict):

            return rag_pipeline(inputs["question"])

        results = self.client.evaluate(
            target,
            data=self.dataset_name,
            evaluators=[
                self.correctness,
                self.answer_relevance,
                self.faithfulness,
                self.retrieval_relevance
            ],
            experiment_prefix="RAG-Evaluation",
            metadata={
                "evaluation_type": "RAG",
                "reranking": True
            }
        )

        logger.info("RAG evaluation completed")

        return results
